# Lab 5 — sklearn Pipeline

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Combine **preprocessing** and **classification** in one `Pipeline`.
2. Use `ColumnTransformer` to apply different transforms to numeric vs categorical columns.
3. Fit and evaluate on the same train/test split as earlier labs (800 / 200).
4. Explain why preprocessing must live **inside** the pipeline (no data leakage).

> **Checkpoints:** steps `['preprocess', 'clf']` · test accuracy ≈ **0.64** · train 800 / test 200

<!-- cisco-day03-lab05-expanded-2026 -->

**Day 3 flow:** [Lab 1 — Probability](lab01_probability_exercises.ipynb) → [Lab 2 — Logistic regression](lab02_logistic_regression.ipynb) → [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb) → [Lab 4 — ROC & AUC](lab04_roc_auc.ipynb) → **Lab 5 (you are here) — sklearn Pipeline** → [Lab 6 — SHAP](lab06_shap_interpretability.ipynb)

## Why a Pipeline?

| Problem | Without Pipeline | With Pipeline |
|---------|------------------|---------------|
| **Data leakage** | Fit scaler on full data, then split | Fit scaler on **train only** inside `pipe.fit()` |
| **Deployment** | Remember transform order by hand | `pipe.predict(new_row)` applies everything |
| **Cross-validation** | Easy to leak labels across folds | `cross_val_score(pipe, X, y)` is safe |

---

## 1. Load data and define feature groups

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
CATEGORICAL_FEATURES = ["grade", "term"]

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df[feature_cols]
y = df["default"]

print(f"shape: {X.shape}")
print(f"default rate: {y.mean():.3f}")
display(X.head(3))


| Type | Columns |
|------|--------|
| Numeric | `loan_amnt`, `int_rate`, `annual_inc`, `dti`, `installment` |
| Categorical | `grade`, `term` |

### 1b. Cardinality of categoricals

In [ ]:
for col in CATEGORICAL_FEATURES:
    print(f"{col}: {X[col].nunique()} unique →", X[col].value_counts().to_dict())


---

## 2. Train/test split (same as Labs 2–4)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}, test size: {len(X_test)}")
print(f"train default rate: {y_train.mean():.3f}")
print(f"test default rate: {y_test.mean():.3f}")


---

## 3. Build `ColumnTransformer`

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

preprocessor


`handle_unknown='ignore'` prevents errors when a new category appears at scoring time.

---

## 4. Wrap in `Pipeline` and fit

In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipe.fit(X_train, y_train)
print(f"pipeline steps: {[name for name, _ in pipe.steps]}")


---

## 5. Predict and measure accuracy

In [ ]:
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"test accuracy: {accuracy:.4f}")
print(f"sample predictions (first 5): {y_pred[:5].tolist()}")


Adding categoricals (`grade`, `term`) typically lifts accuracy over the numeric-only model (~0.59).

---

## 6. Peek inside transformed features

In [ ]:
X_train_transformed = pipe.named_steps["preprocess"].transform(X_train)
print(f"transformed train shape: {X_train_transformed.shape}")

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
all_names = list(NUMERIC_FEATURES) + list(cat_names)
print(f"feature count after encoding: {len(all_names)}")
print("sample encoded names:", all_names[:8], "...")


### 6b. One row through the pipeline

In [ ]:
sample_row = X_test.iloc[[0]]
raw_pred = pipe.predict_proba(sample_row)[0, 1]
print(f"first test row P(default): {raw_pred:.4f}")


---

## 7. Numeric-only baseline

In [ ]:
pipe_numeric = Pipeline(
    steps=[
        ("preprocess", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipe_numeric.fit(X_train[NUMERIC_FEATURES], y_train)
acc_numeric = accuracy_score(y_test, pipe_numeric.predict(X_test[NUMERIC_FEATURES]))

print(f"numeric-only accuracy: {acc_numeric:.4f}")
print(f"full pipeline accuracy: {accuracy:.4f}")
print(f"lift from categoricals: {accuracy - acc_numeric:+.4f}")


---

## 8. Why leakage matters

In [ ]:
# WRONG pattern (do not use in production):
# scaler_bad = StandardScaler().fit(X[NUMERIC_FEATURES])  # fit on ALL rows including test
# Then split — test statistics leaked into scaler
print("Pipeline fits preprocessors only on X_train inside pipe.fit() — safe.")


### 8b. `cross_val_score` with pipeline

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {cv_scores.round(3)}")
print(f"mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


### 8c. Pipeline `predict_proba` on test

In [ ]:
pipe_proba = pipe.predict_proba(X_test)[:, 1]
print(f"score range: {pipe_proba.min():.3f} – {pipe_proba.max():.3f}")


### 8d. Confusion matrix from pipeline

In [ ]:
from sklearn.metrics import confusion_matrix

cm_pipe = confusion_matrix(y_test, y_pred)
print(cm_pipe)


### 8e. Grade coefficient direction (via one-hot)

In [ ]:
# After fit, inspect logistic coefficients on encoded matrix
n_features = pipe.named_steps["clf"].coef_.shape[1]
print(f"classifier sees {n_features} encoded features")


### 8f. Serialize pipeline with joblib

In [ ]:
import joblib

out_dir = GH_ROOT / "hands-on" / "03-classification-interpretation" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
pkl = out_dir / "loan_default_pipe.joblib"
joblib.dump(pipe, pkl)
loaded = joblib.load(pkl)
print("reload predict match:", (loaded.predict(X_test) == y_pred).all())


### 8g. Single-row scoring API shape

In [ ]:
new_app = X_test.iloc[[10]]
print("input columns:", list(new_app.columns))
print("prediction:", pipe.predict(new_app)[0])
print("P(default):", round(pipe.predict_proba(new_app)[0, 1], 4))


### 8h. Term 60 months vs 36 months default rate

In [ ]:
print(df.groupby("term")["default"].mean().round(3))


### 8i. `get_feature_names_out` full list

In [ ]:
feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
print(f"total features: {len(feat_names)}")
print(list(feat_names))


### 8j. Training accuracy from pipeline

In [ ]:
train_acc = accuracy_score(y_train, pipe.predict(X_train))
print(f"train accuracy: {train_acc:.4f}, test: {accuracy:.4f}")


### 8k. `set_params` — change regularization C

In [ ]:
pipe.set_params(clf__C=0.1)
pipe.fit(X_train, y_train)
acc_c01 = accuracy_score(y_test, pipe.predict(X_test))
print(f"accuracy with C=0.1: {acc_c01:.4f}")
pipe.set_params(clf__C=1.0)
pipe.fit(X_train, y_train)


### 8l. Missing value check

In [ ]:
print("missing in X:", X.isna().sum().sum())


### 8m. Grade default rates in raw data

In [ ]:
print(df.groupby("grade")["default"].mean().round(3))


### 8n. Pipeline steps introspection

In [ ]:
for name, step in pipe.steps:
    print(name, "→", type(step).__name__)


### 8o. Sample transformed row

In [ ]:
row_t = pipe.named_steps["preprocess"].transform(X_test.iloc[[0]])
dense = row_t.toarray() if hasattr(row_t, "toarray") else row_t
print("transformed shape:", dense.shape)
print("first 8 values:", np.round(dense[0, :8], 3))


---

## 9. Try it yourself

List the output feature names after one-hot encoding `grade` and `term`.

In [ ]:
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
print(f"total encoded categoricals: {len(names)}")
print(list(names))


---

## 10. Checkpoint summary

In [ ]:
step_names = [name for name, _ in pipe.steps]
assert step_names == ["preprocess", "clf"]
assert len(X_train) == 800 and len(X_test) == 200
assert abs(accuracy - 0.6350) < 0.02
print("✓ All checkpoint assertions passed")


## Feature selection (course topic)

<!-- cisco-topic-coverage -->

Feature **engineering** creates columns; feature **selection** keeps the most predictive subset.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

num_cols = NUMERIC_FEATURES
cat_cols = CATEGORICAL_FEATURES
X_all = df[num_cols + cat_cols]
y_all = df["default"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X_tr[num_cols], y_tr)
scores = pd.DataFrame({"feature": num_cols, "score": selector.scores_}).sort_values("score", ascending=False)
print("top numeric features by ANOVA F-score:")
print(scores.round(2).to_string(index=False))


---

## Reflection questions

1. What would go wrong if you called `StandardScaler().fit(X)` on the full dataset before `train_test_split`?
2. Why is `handle_unknown='ignore'` useful when deploying to production?
3. How would you add `GridSearchCV` to tune `C` without leaking test data?
4. How much lift did `grade` and `term` add over numeric-only?

**Previous:** [Lab 4 — ROC and AUC](lab04_roc_auc.ipynb)  
**Next:** [Lab 6 — SHAP interpretability](lab06_shap_interpretability.ipynb)